IMLO ASSESSMENT -> 23/05/2024

IMLO neural network

In [23]:

#all import will be hers, and reasoning for them
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import numpy as np
#using the adam optimiser
import torch.optim as optim
from torch.optim import Adam
from torchvision.transforms import transforms
import torch
from torch import nn



compute_transform = transforms.Compose([
    transforms.ToTensor(),
])


training_data = datasets.Flowers102(
    root="data",
    split="train",
    download=True,
    transform=compute_transform
)

mean = 0.
std = 0.
for images, _ in training_data:
    mean += np.mean(images.numpy(), axis=(1, 2))
    std += np.std(images.numpy(), axis=(1, 2))

mean /= len(training_data)
std /= len(training_data)

print("Mean:", mean)
print("Std Deviation:", std)

#calculating the means and standard deviations of my model to
# use for tranformations
data_transforms = transforms.Compose([
    transforms.RandomResizedCrop(128),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(),
    transforms.ToTensor(),
    transforms.Normalize(mean.tolist(), std.tolist())
])


test_transform = transforms.Compose([
    transforms.Resize(128), # The images are going to be 128x128
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize(mean.tolist(), std.tolist())
])

training_data = datasets.Flowers102(
    root="data",
    split="train",
    download=True,
    transform=data_transforms
)

test_data = datasets.Flowers102(
    root="data",
    split="test",
    download=True,
    transform=test_transform
)


train_dataloader = DataLoader(training_data, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=32, shuffle=False)

class Flowers_NN(nn.Module):
    def __init__(self):
        super(Flowers_NN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)

        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)

        self.pool = nn.MaxPool2d(2, 2)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

        self.adaptive_pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(512, 102)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = self.pool(self.relu(self.bn4(self.conv4(x))))
        x = self.pool(self.relu(self.bn5(self.conv5(x))))
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


# Initialise the model
model = Flowers_NN()
input_data = torch.randn(64, 3, 150, 150)


output = model(input_data)

print(output.shape)








Mean: [0.4329607  0.38192013 0.29637718]
Std Deviation: [0.26207143 0.21326582 0.22483535]
torch.Size([64, 102])


In [25]:


loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)



In [26]:

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)

    num_batches = len(dataloader)
    train_loss, correct = 0, 0
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred, y)


        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % (num_batches //10 ) ==0:
            loss = loss.item()
            current = (batch * dataloader.batch_size) + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):

    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0


    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


epochs = 30
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
    scheduler.step()
print("Done!")


Epoch 1
-------------------------------
loss: 5.108075  [   32/ 1020]
loss: 4.748220  [  128/ 1020]
loss: 4.897516  [  224/ 1020]
loss: 4.744934  [  320/ 1020]
loss: 4.794100  [  416/ 1020]
loss: 4.421536  [  512/ 1020]
loss: 4.540519  [  608/ 1020]
loss: 4.838669  [  704/ 1020]
loss: 4.247318  [  800/ 1020]
loss: 4.080318  [  896/ 1020]
loss: 4.359725  [  992/ 1020]
Test Error: 
 Accuracy: 7.9%, Avg loss: 4.013319 

Epoch 2
-------------------------------
loss: 3.881020  [   32/ 1020]
loss: 3.905385  [  128/ 1020]
loss: 3.873466  [  224/ 1020]
loss: 4.108829  [  320/ 1020]
loss: 4.062406  [  416/ 1020]
loss: 4.171787  [  512/ 1020]
loss: 3.692074  [  608/ 1020]
loss: 3.925400  [  704/ 1020]
loss: 3.869073  [  800/ 1020]
loss: 4.018549  [  896/ 1020]
loss: 3.843942  [  992/ 1020]
Test Error: 
 Accuracy: 11.3%, Avg loss: 3.833447 

Epoch 3
-------------------------------
loss: 3.399735  [   32/ 1020]
loss: 3.674988  [  128/ 1020]
loss: 3.760964  [  224/ 1020]
loss: 3.549926  [  320/ 102